In [ ]:
import pathlib

import numpy
import polars
from data_index.iceberg_config import S3TablesCatalogConfig, IcebergTableConfig
from data_index.analysis.tables import IMOS_DATA_LIVE_TABLE
from data_index.analysis.datasets import (
    DATASET,
    DATASET_FILTER,
    get_dataset_objects_df,
    get_dataset_arrow_dataset,
)

In [ ]:
table = IMOS_DATA_LIVE_TABLE.load()

In [ ]:
df = table.scan(
    selected_fields=("bucket", "key", "size", "facility", "last_modified_date",),
    row_filter=(
        "facility == 'Argo'"
    ),
).to_polars()

In [ ]:
dataset: DATASET = "argo"

In [ ]:
dataset_df = (
    get_dataset_objects_df(
        df=df,
        dataset=dataset,
    )
)
dataset_df

In [ ]:
ds = get_dataset_arrow_dataset(dataset=dataset,)

In [ ]:
nc_filenames = polars.DataFrame(data={"filename": dataset_df["key"].str.split("/").list.last().unique().sort()})
parquet_filenames = polars.DataFrame(data={
    "filename": [pathlib.Path(fragment.path).name.removesuffix("-0.parquet") for fragment in ds.get_fragments()]
}).unique() 

In [ ]:
nc_filenames
parquet_filenames

In [ ]:
set(nc_filenames["filename"]) - set(parquet_filenames["filename"])

In [ ]:
dataset_df.filter(
    polars.col("key").str.contains("2902089_prof.nc")
    | polars.col("key").str.contains("2903871_prof.nc")
    | polars.col("key").str.contains("2904021_prof.nc")
    | polars.col("key").str.contains("4903612_prof.nc")
    | polars.col("key").str.contains("6990639_prof.nc")
    | polars.col("key").str.contains("7900602_prof.nc")
    | polars.col("key").str.contains("7902066_prof.nc")
)["key"].to_list()

In [ ]:
ds.schema